# Activity 1.3.4 Sentiment analysis of movie reviews

## Scenario
As part of a market research exercise for a film studio planning a new science-fiction film, you have been tasked with a data science project to research customer feedback on films in a related genre. One question you will be asked to investigate is whether there’s a relationship between the proportion of feedback that is positive and production budgets. Before you compare sentiment scores between films, however, you need to construct a viable preprocessing pipeline and train a model.


## Objective
In this portfolio activity, you will apply what you have learned about data preprocessing for NLP as well as some technique for calculating similarity.


## Assessment criteria

By completing this activity, you will be able to provide evidence that you can:
*   construct an NLP classification model capable of predicting binary sentiment class (positive or negative) with high accuracy
*   perform supporting calculations including cosine similarity to enable broad discussion of the techniques, and evaluate model performance
*   report your findings in language that a non-specialist would understand.


## Activity guidance

1. Install the necessary packages that will be useful in this  activity.
2. Load the dataset sst2 from Hugging Face (https://huggingface.co/datasets/sst2).
3. Create dataframes of the train and validation split.
4. Calculate the cosine similarity of the 5th and 100th sentence within the train split.
5. Calculate the cosine similarity of the 5th and 15,000th sentence within the train split.
6. Calculate the cosine similarity of the 5th and 50,000th sentence within the train split.
7. Comment on the cosine similarity scores.
8. Create a preprocessing function to perform several processing steps as described below to the train and validation texts:
 - Remove any punctuation and html tags.
 - Tokenize the text into tokens.
 - Remove stop words from your text.
 - Perform lemmatisation and stemming on your text (one at a time).

9. Obtain Bag-of-Word and TF-IDF representations for both the train and validation splits.
10. Train a logistic regression model using scikit-learn first with the Bag-of-Words and then TF-IDF, and report the performance of the sentiment classifier.


In [1]:
# 1. Install the necessary packages that will be useful in this  activity.
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
# checking libraries installed
try:
    import datasets
    print("Already installed!")
except ImportError:
    print("Need to install it")

Already installed!


In [3]:
!pip install datasets
!pip install spacy
!pip install beautifulsoup4
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
from datasets import load_dataset
dataset = load_dataset('SetFit/sst2')
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 6920
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 872
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1821
    })
})


In [6]:
# 2. Load the dataset sst2 from Hugging Face (https://huggingface.co/datasets/sst2).
from datasets import load_dataset
dataset = load_dataset('SetFit/sst2')
print(dataset)

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 6920
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 872
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1821
    })
})


In [7]:
# 3. Create dataframes of the train and validation split.
import pandas as pd
train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])
print(train_df.head())
print(val_df.head())

                                                text  label label_text
0  a stirring , funny and finally transporting re...      1   positive
1  apparently reassembled from the cutting-room f...      0   negative
2  they presume their audience wo n't sit still f...      0   negative
3  this is a visually stunning rumination on love...      1   positive
4  jonathan parker 's bartleby should have been t...      1   positive
                                                text  label label_text
0                       one long string of cliches .      0   negative
1  if you 've ever entertained the notion of doin...      0   negative
2  k-19 exploits our substantial collective fear ...      0   negative
3  it 's played in the most straight-faced fashio...      0   negative
4  there is a fabric of complex ideas here , and ...      1   positive


In [9]:
# 4. Calculate the cosine similarity of the 5th and 100th sentence within the train split.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer()
train_tfidf = tfidf_vectorizer.fit_transform(train_df['text'])
similarity_matrix = cosine_similarity(train_tfidf)
similarity_5_100 = cosine_similarity(train_tfidf[4], train_tfidf[99])[0][0]
print(f"Cosine similarity between the 5th and 100th sentence in the train split: {similarity_5_100}")

Cosine similarity between the 5th and 100th sentence in the train split: 0.0


In [10]:
print(train_df['text'].iloc[4])
print(train_df['text'].iloc[99])

jonathan parker 's bartleby should have been the be-all-end-all of the modern-office anomie films .
an inexperienced director , mehta has much to learn .


In [11]:
# 5. Calculate the cosine similarity of the 5th and 15,000th sentence within the train split.
similarity_5_15000 = cosine_similarity(train_tfidf[4], train_tfidf[14999])[0][0]
print(f"Cosine similarity between the 5th and 15,000th sentence in the train split: {similarity_5_15000}")

IndexError: index (14999) out of range

In [13]:
from datasets import load_dataset
dataset_glue = load_dataset('nyu-mll/glue', 'sst2')
print(dataset_glue)

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [14]:
# Step 2: Load dataset
from datasets import load_dataset
dataset = load_dataset('nyu-mll/glue', 'sst2')

# Step 3: Create dataframes
import pandas as pd
train_df = pd.DataFrame(dataset['train'])
val_df = pd.DataFrame(dataset['validation'])

# Step 4: Cosine similarity (5th vs 100th)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer()
train_tfidf = tfidf_vectorizer.fit_transform(train_df['sentence'])
similarity_5_100 = cosine_similarity(train_tfidf[4], train_tfidf[99])[0][0]
print(f"Cosine similarity between the 5th and 100th sentence: {similarity_5_100}")

Cosine similarity between the 5th and 100th sentence: 0.011603568769901929


In [17]:
print(train_df['sentence'].iloc[4])
print(train_df['sentence'].iloc[99])

on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
acted and directed , it 's clear that washington most certainly has a new career ahead of him 


In [15]:
# Step 5: Cosine similarity (5th vs 15,000th)
similarity_5_15000 = cosine_similarity(train_tfidf[4], train_tfidf[14999])[0][0]
print(f"Cosine similarity between the 5th and 15,000th sentence: {similarity_5_15000}")

Cosine similarity between the 5th and 15,000th sentence: 0.0


In [18]:
print(train_df['sentence'].iloc[4])
print(train_df['sentence'].iloc[14999])

on the worst revenge-of-the-nerds clichés the filmmakers could dredge up 
eloquent 


In [16]:
# 6. Calculate the cosine similarity of the 5th and 50,000th sentence within the train split.
similarity_5_50000 = cosine_similarity(train_tfidf[4], train_tfidf[49999])[0][0]
print(f"Cosine similarity between the 5th and 50,000th sentence: {similarity_5_50000}")

Cosine similarity between the 5th and 50,000th sentence: 0.0


**7. Comment on the cosine similarity scores.**

I switched datasets between Steps 4-6, as the original dataset (SetFit/sst2)
only had 6,920 train rows, while the activity instructions require comparing
sentences up to index 50,000. I used nyu-mll/glue (sst2 subset) instead,
which has 67,349 train rows.

Results:
- 5th vs 100th: 0.011
- 5th vs 15,000th: 0.0
- 5th vs 50,000th: 0.0

All three scores are very low, which is expected for short, randomly-selected
movie review snippets covering different films and topics. The 5th vs 100th
comparison shows a small non-zero score (0.011), but on closer inspection
this overlap comes from a shared stop word ("the") rather than any genuine
topical similarity - this is noise rather than a meaningful signal.

The 5th vs 15,000th and 5th vs 50,000th comparisons returned exactly 0.0,
indicating no shared vocabulary at all between these sentence pairs.

This highlights why stop word removal (Step 8) is an important preprocessing
step: without it, common words like "the" can create misleadingly non-zero
similarity scores that don't reflect actual content overlap.

In [19]:
# 8. Create a preprocessing function to perform several processing steps as described below to the train and validation texts:
# a. Remove any punctuation and html tags.
import re
from bs4 import BeautifulSoup
import string

def remove_html_punctuation(text):
    soup = BeautifulSoup(text, 'html.parser')
    text = soup.get_text()
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [20]:
# 8b. Tokenize the text into tokens.
def tokenize_text(text):
    tokens = word_tokenize(text)
    return tokens

In [21]:
# 8c. Remove stop words from your text.
def remove_stop_words(tokens):
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    return filtered_tokens

In [23]:
# 8d. Perform lemmatisation and stemming on your text (one at a time).
def stem_tokens(tokens):
    stemmer = PorterStemmer()
    return [stemmer.stem(token) for token in tokens]

def lemmatize_tokens(tokens):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in tokens]

In [24]:
print(train_df.columns)
print(val_df.columns)

Index(['sentence', 'label', 'idx'], dtype='object')
Index(['sentence', 'label', 'idx'], dtype='object')


In [26]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [27]:
# step 8 in one code
def preprocess_text(text, use_stemming=True):
    text = remove_html_punctuation(text)
    tokens = tokenize_text(text)
    tokens = remove_stop_words(tokens)

    if use_stemming:
        tokens = stem_tokens(tokens)
    else:
        tokens = lemmatize_tokens(tokens)

    return ' '.join(tokens)

# Apply with stemming
train_df['stemmed'] = train_df['sentence'].apply(lambda x: preprocess_text(x, use_stemming=True))
val_df['stemmed'] = val_df['sentence'].apply(lambda x: preprocess_text(x, use_stemming=True))

# Apply with lemmatization
train_df['lemmatized'] = train_df['sentence'].apply(lambda x: preprocess_text(x, use_stemming=False))
val_df['lemmatized'] = val_df['sentence'].apply(lambda x: preprocess_text(x, use_stemming=False))

In [28]:
train_df[['sentence', 'stemmed', 'lemmatized']].head()

,sentence,stemmed,lemmatized
0,hide new secretions from the parental units,hide new secret parent unit,hide new secretion parental unit
1,"contains no wit , only labored gags",contain wit labor gag,contains wit labored gag
2,that loves its characters and communicates som...,love charact commun someth rather beauti human...,love character communicates something rather b...
3,remains utterly satisfied to remain the same t...,remain utterli satisfi remain throughout,remains utterly satisfied remain throughout
4,on the worst revenge-of-the-nerds clichés the ...,worst revengeofthenerd cliché filmmak could dredg,worst revengeofthenerds clichés filmmaker coul...


Lemmatization worked better for this dataset.


In [30]:
# 9. Obtain Bag-of-Word and TF-IDF representations for both the train and validation splits.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Bag-of-Words
bow_vectorizer = CountVectorizer()
train_bow = bow_vectorizer.fit_transform(train_df['lemmatized'])
val_bow = bow_vectorizer.transform(val_df['lemmatized'])

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
train_tfidf_final = tfidf_vectorizer.fit_transform(train_df['lemmatized'])
val_tfidf_final = tfidf_vectorizer.transform(val_df['lemmatized'])

In [31]:
# Verify the shapes
print("BoW train shape:", train_bow.shape)
print("BoW val shape:", val_bow.shape)
print("TF-IDF train shape:", train_tfidf_final.shape)
print("TF-IDF val shape:", val_tfidf_final.shape)

BoW train shape: (67349, 13398)
BoW val shape: (872, 13398)
TF-IDF train shape: (67349, 13398)
TF-IDF val shape: (872, 13398)


In [32]:
# 10. Train a logistic regression model using scikit-learn first with the Bag-of-Words and then TF-IDF, and report the performance of the sentiment classifier.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

train_labels = train_df['label']
val_labels = val_df['label']

In [33]:
# Model 1: Bag-of-Words
log_reg_bow = LogisticRegression(max_iter=1000)
log_reg_bow.fit(train_bow, train_labels)
bow_preds = log_reg_bow.predict(val_bow)
print("Bag-of-Words Accuracy:", accuracy_score(val_labels, bow_preds))

# Model 2: TF-IDF
log_reg_tfidf = LogisticRegression(max_iter=1000)
log_reg_tfidf.fit(train_tfidf_final, train_labels)
tfidf_preds = log_reg_tfidf.predict(val_tfidf_final)
print("TF-IDF Accuracy:", accuracy_score(val_labels, tfidf_preds))

Bag-of-Words Accuracy: 0.8107798165137615
TF-IDF Accuracy: 0.8119266055045872


Models reached similar accuracy results.
Bag of words got 81.07% and TF-DF 81.19%.